<a href="https://colab.research.google.com/github/fayaaz01/Content-Curator-2.0/blob/main/Client_Side_Decentralized_Feed_Curator_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Client-Side Decentralized Feed Curator
### Working website project — Google Colab

**Based on the mini-project report:** user-defined preferences, local keyword filtering, relevance ranking, duplicate prevention, and a personalized feed.

#### How to run
1. Open this notebook in Google Colab.
2. Run each code cell from top to bottom.
3. In the last cell, click the website link that appears.
4. Try changing preferred keywords and blocked keywords, then click **Apply preferences**.

**Privacy note:** Filtering and ranking run in the browser with JavaScript. Preferences are saved in that browser's `localStorage`. The notebook server only serves the website files. The demo content is built in; live RSS feeds are not automatically fetched because many feeds block browser cross-origin requests.

## 1. Create the website
Run this cell to write the website's HTML, CSS, and JavaScript into `index.html`.

In [1]:
from pathlib import Path

website_html = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>FeedCurator — Your feed, your rules</title>
<style>
:root{--ink:#172033;--muted:#667085;--bg:#f5f7fb;--card:#fff;--line:#e5e9f2;--brand:#4057d6;--brand2:#2437a7;--green:#13795b;--red:#b42318;--shadow:0 8px 24px rgba(24,39,75,.07)}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.55 Inter,ui-sans-serif,system-ui,-apple-system,Segoe UI,Roboto,Arial}
header{background:linear-gradient(120deg,#172554,#4057d6);color:white;padding:25px max(20px,calc((100vw - 1180px)/2));}
.brandrow{display:flex;align-items:center;gap:12px}.logo{width:43px;height:43px;border-radius:14px;background:#ffffff20;display:grid;place-items:center;font-size:23px}
h1{font-size:clamp(24px,3vw,34px);line-height:1.15;margin:0}header p{margin:10px 0 0;color:#e0e7ff;max-width:700px}
.wrap{max-width:1180px;margin:24px auto;padding:0 18px}.stats{display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin-bottom:18px}.stat,.panel,.article{background:var(--card);border:1px solid var(--line);border-radius:16px;box-shadow:var(--shadow)}
.stat{padding:17px 19px}.stat strong{font-size:27px;display:block}.stat span{color:var(--muted);font-size:13px}
.layout{display:grid;grid-template-columns:320px minmax(0,1fr);gap:18px;align-items:start}.panel{padding:20px}.panel h2{font-size:18px;margin:0 0 5px}.sub{color:var(--muted);font-size:13px;margin:0 0 17px}
label{display:block;font-weight:650;font-size:13px;margin:14px 0 6px}input,textarea,select{width:100%;border:1px solid #d0d5dd;border-radius:10px;padding:10px 11px;font:inherit;background:white;color:var(--ink)}textarea{min-height:65px;resize:vertical}
.help{font-size:12px;color:var(--muted);margin-top:5px}.btn{border:0;border-radius:10px;padding:10px 14px;font-weight:700;font:inherit;cursor:pointer}.primary{background:var(--brand);color:white}.primary:hover{background:var(--brand2)}.secondary{background:#eef2ff;color:#303f9f}.danger{background:#fff1f0;color:var(--red)}.btnrow{display:flex;gap:8px;flex-wrap:wrap;margin-top:15px}
.topline{display:flex;align-items:center;justify-content:space-between;gap:12px;flex-wrap:wrap;margin-bottom:12px}.topline h2{margin:0}.feed{display:grid;gap:12px}.article{padding:17px 18px;transition:transform .15s}.article:hover{transform:translateY(-2px)}.meta{display:flex;gap:8px;align-items:center;flex-wrap:wrap;color:var(--muted);font-size:12px}.tag{background:#eef2ff;color:#3443a4;border-radius:999px;padding:3px 9px;font-weight:700}.score{margin-left:auto;background:#ecfdf3;color:#067647;border-radius:999px;padding:3px 9px;font-weight:700}.article h3{font-size:17px;line-height:1.35;margin:9px 0 6px}.article p{margin:0;color:#475467}.article .reason{font-size:12px;color:var(--green);margin-top:10px}.article .actions{margin-top:12px}.article a{color:var(--brand);font-weight:700;text-decoration:none}.empty{padding:35px;text-align:center;color:var(--muted);border:1px dashed #cbd5e1;border-radius:14px}.status{font-size:13px;margin-top:10px;color:var(--green);min-height:20px}.section-title{font-size:14px;font-weight:800;margin-top:22px;padding-top:15px;border-top:1px solid var(--line)}
.addgrid{display:grid;grid-template-columns:1fr 1fr;gap:10px}.addgrid .wide{grid-column:1/-1}.notice{font-size:12px;color:var(--muted);margin-top:18px;padding:12px;background:#f8fafc;border-radius:10px}
footer{text-align:center;color:var(--muted);font-size:12px;padding:25px}
@media(max-width:820px){.layout{grid-template-columns:1fr}.stats{gap:8px}.stat{padding:13px}.stat strong{font-size:22px}}
@media(max-width:480px){.stats{grid-template-columns:1fr}.addgrid{grid-template-columns:1fr}.addgrid .wide{grid-column:auto}.wrap{padding:0 12px}.panel{padding:16px}}
</style>
</head>
<body>
<header>
 <div class="brandrow"><div class="logo">◈</div><div><h1>FeedCurator</h1><div style="color:#dbe4ff;font-size:13px">Your feed. Your interests. Your rules.</div></div></div>
 <p>A client-side decentralized feed curator that filters unwanted posts and ranks useful content using preferences you control.</p>
</header>
<main class="wrap">
 <section class="stats">
  <div class="stat"><strong id="totalCount">0</strong><span>Items in library</span></div>
  <div class="stat"><strong id="shownCount">0</strong><span>Items in your feed</span></div>
  <div class="stat"><strong id="blockedCount">0</strong><span>Items filtered out</span></div>
 </section>
 <div class="layout">
  <aside class="panel">
   <h2>⚙️ Your preferences</h2><p class="sub">Saved in this browser only.</p>
   <label for="preferred">Topics and keywords I like</label>
   <textarea id="preferred" placeholder="AI, technology, science"></textarea>
   <div class="help">Separate keywords with commas. Matching posts receive higher scores.</div>
   <label for="blocked">Keywords I want to block</label>
   <textarea id="blocked" placeholder="advertisement, spoilers"></textarea>
   <div class="help">Any post containing a blocked keyword is hidden.</div>
   <label for="category">Content category</label>
   <select id="category"><option value="All">All categories</option><option>Technology</option><option>AI Research</option><option>Science</option><option>Education</option><option>Sports</option><option>News</option><option>Community</option></select>
   <label for="query">Search feed</label><input id="query" placeholder="Search titles or descriptions…">
   <div class="btnrow"><button class="btn primary" id="apply">Apply preferences</button><button class="btn secondary" id="reset">Reset demo</button></div>
   <div class="status" id="status" role="status" aria-live="polite"></div>
   <div class="section-title">＋ Add your own content</div>
   <div class="addgrid">
    <div class="wide"><label for="newTitle">Title</label><input id="newTitle" placeholder="A useful article title"></div>
    <div><label for="newCategory">Category</label><select id="newCategory"><option>Technology</option><option>AI Research</option><option>Science</option><option>Education</option><option>Sports</option><option>News</option><option>Community</option></select></div>
    <div><label for="newSource">Source</label><input id="newSource" placeholder="My source"></div>
    <div class="wide"><label for="newDesc">Description</label><textarea id="newDesc" placeholder="Brief summary of the content"></textarea></div>
    <div class="wide"><label for="newLink">Link (optional)</label><input id="newLink" type="url" placeholder="https://example.com/article"></div>
   </div>
   <div class="btnrow"><button class="btn primary" id="addArticle">Add content</button></div>
   <div class="notice">Privacy by design: preferences and items you add are stored in your browser's localStorage. This demo uses sample content; external RSS feeds may require a server-side proxy because of browser CORS restrictions.</div>
  </aside>
  <section>
   <div class="topline"><div><h2>✨ Personalized feed</h2><div class="sub" style="margin:4px 0 0">Relevant items appear first. Your blocked terms are applied before ranking.</div></div>
    <select id="sort" style="width:auto;min-width:155px"><option value="relevance">Most relevant</option><option value="newest">Recently added</option><option value="title">Title A–Z</option></select>
   </div>
   <div id="feed" class="feed" aria-live="polite"></div>
  </section>
 </div>
</main>
<footer>Client-Side Decentralized Feed Curator · Mini Project · HTML, CSS, JavaScript and browser localStorage</footer>
<script>
const demoItems = [
{id:1,title:"Latest AI Developments: What to Know",description:"A simple overview of recent artificial intelligence tools, research, and responsible AI applications.",category:"AI Research",source:"Demo Research Desk",date:"2026-10-29",link:"https://example.com/ai-developments"},
{id:2,title:"Technology News: Open Web Standards",description:"How open standards and interoperable web technologies help developers build more flexible products.",category:"Technology",source:"Demo Tech Journal",date:"2026-10-28",link:"https://example.com/open-web"},
{id:3,title:"New Research in Climate Science",description:"Researchers study climate data and practical approaches to environmental monitoring.",category:"Science",source:"Demo Science Brief",date:"2026-10-27",link:"https://example.com/climate-science"},
{id:4,title:"Advertisement: Limited-Time Shopping Offer",description:"Promotional advertisement with a limited-time deal and sponsored product placement.",category:"News",source:"Demo Promotions",date:"2026-10-26",link:"https://example.com/promo"},
{id:5,title:"Sports Updates: Weekend Match Roundup",description:"A roundup of match results, team performances, and upcoming sports events.",category:"Sports",source:"Demo Sports Desk",date:"2026-10-25",link:"https://example.com/sports"},
{id:6,title:"Learning Python: A Beginner's Guide",description:"A practical introduction to Python programming, variables, functions, and small projects.",category:"Education",source:"Demo Learning Hub",date:"2026-10-24",link:"https://example.com/python"},
{id:7,title:"Why Privacy-Friendly Software Matters",description:"Explore local-first software, data minimization, user choice, and transparent application design.",category:"Technology",source:"Demo Tech Journal",date:"2026-10-23",link:"https://example.com/privacy"},
{id:8,title:"AI Research: Smaller Models, Useful Results",description:"A look at efficient AI models, evaluation, and research methods for real-world applications.",category:"AI Research",source:"Demo Research Desk",date:"2026-10-22",link:"https://example.com/small-models"},
{id:9,title:"Community Science Workshop Announced",description:"A local learning event invites students to explore science, coding, and hands-on experiments.",category:"Community",source:"Demo Community Board",date:"2026-10-21",link:"https://example.com/workshop"},
{id:10,title:"Breaking News: City Transport Update",description:"A sample public update about transport planning and changes to local travel routes.",category:"News",source:"Demo News Wire",date:"2026-10-20",link:"https://example.com/transport"}
];
const $ = id => document.getElementById(id);
const STORE = "feedcurator_v1";
let state = loadState();
function defaults(){return {preferred:"AI, technology",blocked:"advertisement",category:"All",query:"",sort:"relevance",items:demoItems};}
function loadState(){try{const s=JSON.parse(localStorage.getItem(STORE));if(s&&Array.isArray(s.items))return {...defaults(),...s};}catch(e){}return defaults();}
function saveState(){try{localStorage.setItem(STORE,JSON.stringify(state));}catch(e){}}
function terms(v){return String(v||"").split(",").map(x=>x.trim().toLowerCase()).filter(Boolean);}
function safeUrl(raw){try{const u=new URL(raw);return ["http:","https:"].includes(u.protocol)?u.href:"#";}catch(e){return "#";}}
function curate(){
 const pref=terms(state.preferred), blocked=terms(state.blocked), q=(state.query||"").trim().toLowerCase();
 let rejected=0, accepted=[];
 for(const item of state.items){
  const hay=(item.title+" "+item.description+" "+item.category+" "+item.source).toLowerCase();
  const bad=blocked.find(k=>hay.includes(k));
  if(bad){rejected++;continue;}
  if(state.category!=="All"&&item.category!==state.category)continue;
  if(q&&!hay.includes(q))continue;
  const matches=pref.filter(k=>hay.includes(k));
  accepted.push({...item,score:matches.length,matched:matches});
 }
 if(state.sort==="title")accepted.sort((a,b)=>a.title.localeCompare(b.title));
 else if(state.sort==="newest")accepted.sort((a,b)=>(b.date||"").localeCompare(a.date||""));
 else accepted.sort((a,b)=>b.score-a.score || (b.date||"").localeCompare(a.date||""));
 return {accepted,rejected};
}
function render(){
 const {accepted,rejected}=curate();
 $("totalCount").textContent=state.items.length;
 $("shownCount").textContent=accepted.length;
 $("blockedCount").textContent=rejected;
 const feed=$("feed");
 if(!accepted.length){feed.innerHTML='<div class="empty"><div style="font-size:28px">⌕</div><strong>No matching content</strong><br>Try another category, search term, or preference.</div>';return;}
 feed.innerHTML="";
 for(const item of accepted){
  const card=document.createElement("article");card.className="article";
  const meta=document.createElement("div");meta.className="meta";
  const cat=document.createElement("span");cat.className="tag";cat.textContent=item.category||"General";
  const source=document.createElement("span");source.textContent=(item.source||"User added")+" · "+(item.date||"Added just now");
  const score=document.createElement("span");score.className="score";score.textContent="Relevance "+item.score;
  meta.append(cat,source,score);
  const title=document.createElement("h3");title.textContent=item.title;
  const desc=document.createElement("p");desc.textContent=item.description;
  const reason=document.createElement("div");reason.className="reason";reason.textContent=item.matched.length?"✓ Matched: "+item.matched.join(", "):"○ No preferred keyword match — shown with a lower score";
  const actions=document.createElement("div");actions.className="actions";
  const a=document.createElement("a");a.href=safeUrl(item.link||"");a.target="_blank";a.rel="noopener noreferrer";a.textContent="Open source ↗";
  if(a.href==="#"||!item.link){a.removeAttribute("target");a.textContent="Sample item";}
  actions.append(a);card.append(meta,title,desc,reason,actions);feed.append(card);
 }
}
function syncInputs(){ $("preferred").value=state.preferred;$("blocked").value=state.blocked;$("category").value=state.category;$("query").value=state.query;$("sort").value=state.sort;}
$("apply").addEventListener("click",()=>{state.preferred=$("preferred").value;state.blocked=$("blocked").value;state.category=$("category").value;state.query=$("query").value;state.sort=$("sort").value;saveState();render();$("status").textContent="Preferences applied and saved in this browser.";});
$("sort").addEventListener("change",()=>{state.sort=$("sort").value;saveState();render();});
$("category").addEventListener("change",()=>{state.category=$("category").value;saveState();render();});
$("query").addEventListener("input",()=>{state.query=$("query").value;render();});
$("addArticle").addEventListener("click",()=>{
 const title=$("newTitle").value.trim(),description=$("newDesc").value.trim(),source=$("newSource").value.trim()||"User added";
 const link=$("newLink").value.trim();
 if(!title||!description){$("status").textContent="Please enter both a title and a description.";return;}
 if(link&&!/^https?:\/\//i.test(link)){$("status").textContent="Please enter a valid http:// or https:// link.";return;}
 state.items.unshift({id:Date.now(),title,description,category:$("newCategory").value,source,date:new Date().toISOString().slice(0,10),link});
 saveState();render();$("newTitle").value="";$("newDesc").value="";$("newSource").value="";$("newLink").value="";$("status").textContent="Content added to your local feed.";
});
$("reset").addEventListener("click",()=>{if(confirm("Reset preferences and restore the original demo items?")){state=defaults();saveState();syncInputs();render();$("status").textContent="Demo reset."; }});
syncInputs();render();
</script>
</body>
</html>
"""

Path("/content/index.html").write_text(website_html, encoding="utf-8")
print("Website created: /content/index.html")


Website created: /content/index.html


## 2. Start the website
Run this cell after the first cell. It starts a small web server inside the Colab runtime and displays a clickable website link. Keep the notebook runtime running while you use the site.

In [2]:
import os
import threading
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from functools import partial
from IPython.display import display, HTML

PORT = 8000

# Avoid starting duplicate servers if this cell is run more than once.
if "feedcurator_server" not in globals():
    handler = partial(SimpleHTTPRequestHandler, directory="/content")
    feedcurator_server = ThreadingHTTPServer(("0.0.0.0", PORT), handler)
    feedcurator_thread = threading.Thread(target=feedcurator_server.serve_forever, daemon=True)
    feedcurator_thread.start()

try:
    from google.colab.output import eval_js
    website_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    display(HTML(f"""
    <div style="font-family:Arial,sans-serif;padding:18px;border:1px solid #dbe3f0;border-radius:12px">
      <h3 style="margin-top:0">Your website is running</h3>
      <p>Click the button to open FeedCurator in a new tab.</p>
      <p><a href="{website_url}" target="_blank" style="display:inline-block;background:#4057d6;color:white;padding:12px 18px;border-radius:8px;text-decoration:none;font-weight:bold">Open FeedCurator website ↗</a></p>
      <p style="font-size:13px;color:#667085">The link works while this Colab runtime is active. This is a development preview, not permanent public hosting.</p>
    </div>
    """))
except Exception as e:
    print("Server started on port 8000.")
    print("In Google Colab, use the port preview for port 8000 to open the website.")


## Features included
- Preferred keywords and blocked keywords
- Rule-based filtering and relevance scoring
- Category filter, search, and sorting
- Add your own content items
- Duplicate-free starter dataset
- Browser `localStorage` for preferences and user-added items
- Responsive layout for desktop and mobile

## Important project note
This is a working **prototype** based on the report's HTML/CSS/JavaScript and local-storage design. It uses sample content and user-added items. Live RSS/Atom fetching is not included in this first version because browser cross-origin restrictions vary by source. A later version can add a small RSS proxy or a supported feed API if required.